# Lesson 28: Time series, recommender systems and graphs

*Companion notebook for [the lesson page](../lessons/28-time-series-recsys-graphs.html). Worked solutions are in `solutions/28-time-series-recsys-graphs-solutions.ipynb`; try each exercise first.*

Three kinds of data that break the independent-rows assumption: forecast with honest baselines and walk-forward backtests, build and evaluate recommenders from matrix factorisation to two-tower retrieval, and learn on graphs with PageRank, node embeddings and a graph neural network written from scratch.

**Runs on:** Any computer (CPU), offline: the data ships with statsmodels and networkx or is generated in code. One optional cell downloads MovieLens 100k (5 MB). The whole notebook runs in a few minutes.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn scipy statsmodels networkx torch

## Data that is not a bag of independent rows

Lessons 03 and 04 treated data as a table of independent rows: shuffle them, split them, and each row is a fresh draw from the same distribution. A large share of production machine learning breaks that assumption in one of three ways.

## Time series: what makes them different

A **time series** is a sequence of measurements at regular intervals, $y_1, y_2, \dots, y_T$. Four properties shape everything you do with one.

Our running example is the concentration of carbon dioxide in the air at Mauna Loa, Hawaii, which ships with statsmodels. It comes as weekly readings from 1958 to 2001 with gaps, so we average to months and interpolate the five empty ones.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL

raw = sm.datasets.co2.load_pandas().data["co2"]    # weekly, Mauna Loa, 1958-2001
y = raw.resample("MS").mean().interpolate()        # monthly means, fill 5 empty months
print(len(y), "months from", y.index[0].date(), "to", y.index[-1].date())
for lag in [1, 6, 12]:
    print(f"lag {lag:>2}: raw {y.autocorr(lag):.3f}   "
          f"differenced {y.diff().autocorr(lag):+.3f}")

parts = STL(y, period=12).fit()                    # trend + seasonal + remainder
by_month = parts.seasonal.groupby(y.index.month).mean()
print(f"trend {parts.trend.iloc[0]:.1f} to {parts.trend.iloc[-1]:.1f} ppm, "
      f"remainder std {parts.resid.std():.2f} ppm")
print(f"seasonal peak: month {by_month.idxmax()} ({by_month.max():+.1f}), "
      f"trough: month {by_month.idxmin()} ({by_month.min():+.1f})")

Always look at a series before modelling it. The four panels are the data, the trend, the seasonal component and the remainder.

In [ ]:
import matplotlib.pyplot as plt

fig = parts.plot()
fig.set_size_inches(9, 7)
plt.show()

## Baselines and metrics

Three baselines cost nothing: **naive** (repeat the last value), **seasonal naive** (repeat the value from the same season last year) and the **moving average** (repeat the mean of a recent window). They are surprisingly hard to beat, and a model that cannot beat them is not worth deploying.

MAE and RMSE are in the units of the data; RMSE punishes large errors more. MAPE, the mean absolute percentage error, is popular in business reports and full of traps: it is undefined when the actual value is zero and explodes near zero, it depends on the level of the series, and it punishes over-forecasts more than under-forecasts (an under-forecast can be at most 100% wrong, an over-forecast has no limit). The **MASE** (mean absolute scaled error) avoids all of this by dividing your MAE by the in-sample MAE of the seasonal naive forecast:

$$
\text{MASE} = \frac{\frac{1}{h}\sum_{t=T+1}^{T+h} |y_t - \hat{y}_t|}{\frac{1}{T-m}\sum_{t=m+1}^{T} |y_t - y_{t-m}|}
$$

Here $m$ is the season length, 12 for monthly data. Below 1 you beat the benchmark, above 1 you lose to it, and because it has no units you can average it across many series.

In [ ]:
def mae(a, f):  return np.mean(np.abs(a - f))
def rmse(a, f): return np.sqrt(np.mean((a - f) ** 2))
def mape(a, f): return 100 * np.mean(np.abs((a - f) / a))
def mase(a, f, train, m=12):
    scale = np.mean(np.abs(train[m:] - train[:-m]))  # in-sample seasonal naive MAE
    return mae(a, f) / scale

h = 24                                             # hold out the last two years
train, test = y.values[:-h], y.values[-h:]
forecasts = {
    "naive":          np.repeat(train[-1], h),
    "moving avg 12":  np.repeat(train[-12:].mean(), h),
    "seasonal naive": np.tile(train[-12:], h // 12),
}
for name, f in forecasts.items():
    print(f"{name:<15} MAE {mae(test, f):4.2f}  RMSE {rmse(test, f):4.2f}  "
          f"MAPE {mape(test, f):4.2f}%  MASE {mase(test, f, train):4.2f}")

## Backtesting: walk forward, never shuffle

One holdout period is one sample, and a lucky or unlucky year can mislead you. A **backtest**, or walk-forward validation, repeats the honest experiment several times: pick a forecast origin, train on everything before it, forecast the next $h$ steps, score them, move the origin forward and repeat.

Random K-fold cross-validation, the default from lesson 03, is wrong here. Shuffled folds let the model train on months either side of each test month, so it interpolates instead of forecasting, and features such as rolling means quietly contain the future. This is temporal leakage, and it is the most common way forecasting projects fool themselves. scikit-learn's `TimeSeriesSplit` produces walk-forward folds; for full control, write the loop yourself.

In [ ]:
def backtest(y, forecast_fn, h=12, n_folds=6):
    """Walk forward: at each origin, train on all earlier data, forecast h steps."""
    scores = []
    for k in range(n_folds, 0, -1):
        cut = len(y) - k * h                       # forecast origin of this fold
        train, test = y.iloc[:cut], y.iloc[cut:cut + h]
        f = np.asarray(forecast_fn(train, h))
        scores.append(mase(test.values, f, train.values))
    return np.mean(scores)

def seasonal_naive(train, h):
    return np.resize(train.values[-12:], h)        # repeat the last year

def seasonal_naive_drift(train, h):                # ...plus last year's rise
    rise = train.values[-12:].mean() - train.values[-24:-12].mean()
    return seasonal_naive(train, h) + rise * (np.arange(h) // 12 + 1)

for name, fn in [("seasonal naive", seasonal_naive),
                 ("seasonal naive + drift", seasonal_naive_drift)]:
    print(f"{name:<24} MASE {backtest(y, fn):.3f}")

## Exponential smoothing and ARIMA

### Exponential smoothing

**Simple exponential smoothing** forecasts with a weighted average of past values whose weights shrink exponentially with age:

$$
\hat{y}_{t+1} = \alpha\, y_t + (1-\alpha)\,\hat{y}_t, \qquad 0 < \alpha < 1
$$

### ARIMA

**ARIMA**(p, d, q) first differences the series $d$ times to make it stationary, then models each value as a linear function of the previous $p$ values (the autoregressive part) and the previous $q$ forecast errors (the moving-average part). Seasonal ARIMA, or SARIMA, adds the same three ingredients at the seasonal lag, written $(P, D, Q)_m$. The augmented Dickey-Fuller (ADF) test checks for stationarity. Its null hypothesis is that the series is not stationary, so after differencing you want a small p-value.

In [ ]:
import warnings
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings("ignore", module="statsmodels")   # convergence chatter

for name, s in [("raw", y), ("differenced", y.diff(12).diff().dropna())]:
    print(f"ADF p-value, {name}: {adfuller(s)[1]:.3f}")

def holt_winters(train, h):
    fit = ExponentialSmoothing(train.values, trend="add", seasonal="add",
                               seasonal_periods=12).fit()
    return fit.forecast(h)

def sarima(train, h):
    fit = SARIMAX(train.values, order=(1, 1, 1),
                  seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    return fit.forecast(h)

for name, fn in [("Holt-Winters", holt_winters), ("SARIMA", sarima)]:
    print(f"{name:<24} MASE {backtest(y, fn):.3f}")

### Prediction intervals

A point forecast hides its uncertainty, and most decisions (how much stock to order, how much capacity to buy) need a range. SARIMA derives intervals from its error model. Check them in the backtest: an 80% interval should contain about 80% of the actual values.

In [ ]:
hits, widths = [], []
for k in range(6, 0, -1):                          # the same six folds
    cut = len(y) - k * 12
    fit = SARIMAX(y.values[:cut], order=(1, 1, 1),
                  seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    lo, hi = fit.get_forecast(12).conf_int(alpha=0.2).T    # 80% interval
    actual = y.values[cut:cut + 12]
    hits += list((actual >= lo) & (actual <= hi))
    widths.append(np.mean(hi - lo))
print(f"80% intervals covered {np.mean(hits):.0%} of {len(hits)} months, "
      f"mean width {np.mean(widths):.2f} ppm")

The last fold of the backtest: three years of history, the forecast and its 80% interval.

In [ ]:
fc = fit.get_forecast(12)
lo, hi = fc.conf_int(alpha=0.2).T
idx = y.index[cut:cut + 12]
plt.figure(figsize=(9, 3.5))
plt.plot(y.index[cut - 36:cut + 12], y.values[cut - 36:cut + 12], label="actual")
plt.plot(idx, fc.predicted_mean, label="SARIMA forecast")
plt.fill_between(idx, lo, hi, alpha=0.3, label="80% interval")
plt.legend()
plt.show()

## Forecasting with machine learning

Any forecasting problem can be recast as supervised regression. Each row is a time point, the target is the value to predict, and the features are **lags** (past values), rolling statistics, **calendar features** (month, day of week, holidays) and outside drivers such as price or weather. Then the gradient boosting of lesson 04 applies. Two traps are waiting.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

X = pd.DataFrame({"t": np.arange(len(y)), "month": y.index.month})
gbr = HistGradientBoostingRegressor(random_state=0)
for name, cv in [("shuffled K-fold", KFold(5, shuffle=True, random_state=0)),
                 ("walk-forward", TimeSeriesSplit(5, test_size=12))]:
    err = -cross_val_score(gbr, X, y, cv=cv, scoring="neg_mean_absolute_error")
    print(f"{name:<16} MAE {err.mean():.2f} ppm")

The fix is to predict something stationary, here the year-on-year change, and add it back to last year's value. Every feature must also exist at forecast time: to forecast 12 months ahead, every lag must be at least 12 months old. One model on such features for every horizon is the **direct** strategy; the alternative, **recursive** forecasting, predicts one step and feeds the prediction back in as a lag.

In [ ]:
def lag_features(s):
    """Features for month t built only from values at t-12 or earlier."""
    d = s - s.shift(12)                            # year-on-year change
    X = pd.DataFrame({
        "month":    s.index.month,
        "d_lag12":  d.shift(12),                   # the change a year earlier
        "d_lag13":  d.shift(13),
        "d_mean12": d.shift(12).rolling(12).mean(),   # average change that year
    }, index=s.index)
    return X, d

def gbm(train, h):                                 # direct forecast, needs h <= 12
    future = pd.date_range(train.index[-1], periods=h + 1, freq="MS")[1:]
    s = pd.concat([train, pd.Series(np.nan, index=future)])
    X, d = lag_features(s)
    rows = X.notna().all(axis=1) & d.notna()       # complete rows, all in the past
    model = HistGradientBoostingRegressor(learning_rate=0.05, random_state=0)
    model.fit(X[rows], d[rows])
    return s.shift(12).iloc[-h:].values + model.predict(X.iloc[-h:])

print(f"{'gradient boosting':<24} MASE {backtest(y, gbm):.3f}")

## Many series, deep models and pretrained forecasters

Real forecasting usually means thousands of series: every product in every store. A **local** approach fits one model per series. A **global** approach trains one model on all of them, with each series scaled by its own level and its attributes as features, so a new product borrows patterns from similar old ones. When series form a hierarchy (store, region, country), the forecasts should add up, which reconciliation methods enforce.

### Hands-on: backtest a harder series

1. Load the sea-surface temperatures that ship with statsmodels, `sm.datasets.elnino.load_pandas().data` (one row per year from 1950, one column per month), and reshape them into a monthly Series with a `"MS"` date index. You should get 732 months.
2. Backtest seasonal naive, seasonal naive with drift, Holt-Winters, SARIMA and the gradient-boosting forecaster at $h = 12$. Does anything beat seasonal naive?
3. Write a `climatology` forecaster that predicts each calendar month's mean over all training years, and backtest it too.
4. Backtest seasonal naive and Holt-Winters at $h$ = 1, 3 and 6, keeping 72 test months (`n_folds=72 // h`). How does the gap between them change with the horizon?

In [ ]:
# Your code here

## Recommender systems: the problem

A recommender has users, items and a log of interactions between them, and for each user it must produce a short ranked list of items they will value. Picture the log as a matrix with a row per user and a column per item: almost every cell is empty. MovieLens 100k, a classic research dataset, fills 6.3% of its cells, and real catalogues are far sparser.

### The multi-stage architecture

Scoring millions of items with a large model on every request is too slow, so production systems work as a funnel. **Candidate generation** uses cheap methods, often several at once, to pull a few hundred to a few thousand plausible items. **Ranking** scores those with a heavier model and richer features. **Re-ranking** applies diversity, freshness and business rules to the final list. Covington and colleagues described YouTube's recommender in these terms in 2016: a candidate generation network followed by a ranking network.

## A synthetic ratings matrix

We generate data with a known structure so we can check what each model recovers. 1,000 users and 500 items each get a hidden vector of 4 taste numbers, items cluster by genre, and a user's true affinity for an item is a bias plus the dot product of their vectors. As in real life, users mostly rate items that are popular and items they expect to like, and about 10% of the items are released late in the year.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n_users, n_items, n_genres, rank = 1000, 500, 8, 4

genre = rng.integers(0, n_genres, n_items)         # each item has one genre
centres = rng.normal(0, 1, (n_genres, rank))
Q_true = centres[genre] + 0.5 * rng.normal(0, 1, (n_items, rank))
P_true = rng.normal(0, 1, (n_users, rank)) / np.sqrt(rank)
taste = (3.4 + rng.normal(0, 0.3, (n_users, 1)) + rng.normal(0, 0.4, n_items)
         + 0.8 * P_true @ Q_true.T)                # true affinity, users x items

# who rates what: popular items, and items you like, are more likely to be seen
log_pop = rng.normal(0, 1, n_items)
gumbel = -np.log(-np.log(rng.random((n_users, n_items))))
order = np.argsort(-(log_pop + taste + gumbel), axis=1)
n_rated = rng.integers(10, 80, n_users)
u_idx = np.repeat(np.arange(n_users), n_rated)
i_idx = np.concatenate([order[u, :n] for u, n in enumerate(n_rated)])

# most items exist from day 0; about 10% are released after day 300
release = np.where(rng.random(n_items) < 0.1, rng.uniform(300, 330, n_items), 0.0)
day = release[i_idx] + (365 - release[i_idx]) * rng.random(len(u_idx))
noise = rng.normal(0, 0.5, len(u_idx))
rating = np.clip(np.round(taste[u_idx, i_idx] + noise), 1, 5).astype(int)
df = pd.DataFrame({"user": u_idx, "item": i_idx, "rating": rating,
                   "day": day.round(1)}).sort_values("day", ignore_index=True)

print(f"{len(df)} ratings, density {len(df) / (n_users * n_items):.1%}")
print("count of each rating 1-5:", np.bincount(df.rating, minlength=6)[1:])
top = df.item.value_counts().iloc[:n_items // 10].sum() / len(df)
print(f"the top 10% of items receive {top:.0%} of all ratings")

Plot ratings per item, most popular first: a few items get many ratings and most get few.

In [ ]:
import matplotlib.pyplot as plt

counts = df.item.value_counts().values
plt.figure(figsize=(8, 3))
plt.plot(counts)
plt.xlabel("item, most rated first")
plt.ylabel("number of ratings")
plt.title("The long tail")
plt.show()

## Baselines first

Split by time: train on everything before day 300 and test on what came after. A random split would let the model learn from a user's future to predict their past, and it would hide the new items that a live system must handle.

The simplest baseline predicts the global mean $\mu$. The next adds a user bias $b_u$ (some people rate everything highly) and an item bias $b_i$ (some items are simply better):

$$
\hat{r}_{ui} = \mu + b_u + b_i, \qquad b_i = \frac{\sum_{u} (r_{ui} - \mu)}{n_i + \lambda}
$$

The $\lambda$ in the denominator shrinks the bias of an item with few ratings towards zero, so a single five-star rating does not make it the best item in the catalogue.

In [ ]:
train, test = df[df.day < 300], df[df.day >= 300]  # split by time, not at random
warm = test.user.isin(train.user) & test.item.isin(train.item)
print(f"train {len(train)}, test {len(test)}: {(~warm).sum()} test ratings are "
      f"for {test.item[~warm].nunique()} items released after the cut")
test_w = test[warm]                                # what CF models can score

def rmse(a, b):
    return np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

mu, lam = train.rating.mean(), 5.0                 # lam shrinks small groups to 0
def shrunk_mean(r, keys):
    g = r.groupby(keys)
    return g.sum() / (g.count() + lam)

b_i = shrunk_mean(train.rating - mu, train.item).reindex(range(n_items), fill_value=0)
b_u = shrunk_mean(train.rating - mu - b_i[train.item].values, train.user)
b_u = b_u.reindex(range(n_users), fill_value=0)

def baseline(d):
    return mu + b_u[d.user].values + b_i[d.item].values

print(f"global mean      RMSE {rmse(test_w.rating, mu):.3f}")
print(f"mean + biases    RMSE {rmse(test_w.rating, baseline(test_w)):.3f}")

## Collaborative filtering

**Collaborative filtering** predicts your preferences from other people's: users who agreed in the past tend to agree again. It needs no knowledge of what the items are.

### Neighbourhood methods

User-based filtering finds people similar to you and recommends what they liked. Item-based filtering finds items similar to the ones you rated highly, the "customers who bought this also bought" idea that Amazon made famous. Item-based is the usual choice in practice, because item similarities change slowly and can be precomputed. Below, two items are similar when their columns of residuals (rating minus baseline) point the same way, and a prediction is the similarity-weighted average of your own residuals on the 30 most similar items.

In [ ]:
R = np.zeros((n_users, n_items))                   # residuals after the baseline
M = np.zeros((n_users, n_items))                   # 1 where a rating exists
R[train.user, train.item] = train.rating - baseline(train)
M[train.user, train.item] = 1

norms = np.linalg.norm(R, axis=0) + 1e-9
S = (R.T @ R) / np.outer(norms, norms)             # item-item cosine similarity
np.fill_diagonal(S, 0)
kth = -np.sort(-S, axis=1)[:, [29]]                # 30th largest in each row
S30 = np.where(S >= kth, S, 0)                     # keep 30 neighbours per item

# predicted residual = similarity-weighted average of the user's own residuals
cf_resid = (R @ S30.T) / np.maximum(M @ np.abs(S30).T, 1e-9)
cf_pred = baseline(test_w) + cf_resid[test_w.user, test_w.item]
print(f"item-based CF    RMSE {rmse(test_w.rating, cf_pred):.3f}")

### Matrix factorisation

**Matrix factorisation** gives every user a vector $\mathbf{p}_u$ and every item a vector $\mathbf{q}_i$ in a small shared space of taste dimensions, and predicts with their dot product, much like the embeddings of lesson 07:

$$
\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i
$$

Training minimises the squared error over the observed cells only, plus L2 regularisation. Never fill the empty cells with zeros: that teaches the model that unseen means hated.

$$
\min_{P,\,Q,\,b} \sum_{(u,i)\ \text{observed}} \left(r_{ui} - \hat{r}_{ui}\right)^2 + \lambda\left(\lVert\mathbf{p}_u\rVert^2 + \lVert\mathbf{q}_i\rVert^2 + b_u^2 + b_i^2\right)
$$

With stochastic gradient descent, each rating with error $e = r_{ui} - \hat{r}_{ui}$ nudges $\mathbf{p}_u \leftarrow \mathbf{p}_u + \eta\,(e\,\mathbf{q}_i - \lambda\,\mathbf{p}_u)$, and $\mathbf{q}_i$ symmetrically. The alternative, **alternating least squares** (ALS), fixes the item vectors and solves each user's vector exactly as a small ridge regression, then swaps roles; it parallelises well and has a widely used variant for implicit feedback. This family of models was central to the Netflix Prize, as Koren, Bell and Volinsky describe.

In [ ]:
def train_mf(train, k=8, epochs=30, lr=0.05, reg=0.05, seed=0):
    """Biased matrix factorisation, mini-batch SGD on the observed cells only."""
    rng = np.random.default_rng(seed)
    u, i = train.user.values, train.item.values
    r = train.rating.values.astype(float)
    mu = r.mean()
    P = 0.1 * rng.normal(size=(n_users, k))        # user vectors
    Q = 0.1 * rng.normal(size=(n_items, k))        # item vectors
    bu, bi = np.zeros(n_users), np.zeros(n_items)
    for epoch in range(epochs):
        for b in np.array_split(rng.permutation(len(r)), len(r) // 256):
            ub, ib = u[b], i[b]
            err = r[b] - (mu + bu[ub] + bi[ib] + (P[ub] * Q[ib]).sum(1))
            gP = err[:, None] * Q[ib] - reg * P[ub]
            gQ = err[:, None] * P[ub] - reg * Q[ib]
            np.add.at(bu, ub, lr * (err - reg * bu[ub]))   # add.at sums repeats
            np.add.at(bi, ib, lr * (err - reg * bi[ib]))
            np.add.at(P, ub, lr * gP)
            np.add.at(Q, ib, lr * gQ)
    return lambda us, its: mu + bu[us] + bi[its] + (P[us] * Q[its]).sum(-1)

mf = train_mf(train)
print(f"MF train RMSE {rmse(train.rating, mf(train.user.values, train.item.values)):.3f}")
print(f"MF test  RMSE {rmse(test_w.rating, mf(test_w.user.values, test_w.item.values)):.3f}")
oracle = np.clip(taste[test_w.user, test_w.item], 1, 5)    # the true taste
print(f"oracle   RMSE {rmse(test_w.rating, oracle):.3f}")

## Evaluating recommendations as rankings

Users never see an RMSE. They see a top-10 list, so evaluate the list. For each test user, call the test items they rated 4 or 5 *relevant*, drop everything they rated in training from the candidates, take the model's top $k$ and count the hits:

- **precision@k**: the fraction of the $k$ recommendations that are relevant;
- **recall@k**: the fraction of the relevant items that made the top $k$;
- **NDCG@k** (normalised discounted cumulative gain): hits count more the higher they appear, divided by the best possible score so that a perfect list scores 1.

$$
\text{DCG@}k = \sum_{j=1}^{k} \frac{\text{rel}_j}{\log_2(j+1)}, \qquad \text{NDCG@}k = \frac{\text{DCG@}k}{\text{ideal DCG@}k}
$$

In [ ]:
def ranking_metrics(scores, train, test, k=10):
    """Mean precision@k, recall@k, NDCG@k. Relevant = test items rated 4 or 5."""
    s = scores.copy()
    s[train.user, train.item] = -np.inf            # never re-recommend
    liked = test[test.rating >= 4].groupby("user").item.apply(set)
    discount = 1 / np.log2(np.arange(2, k + 2))    # 1 / log2(rank + 1)
    prec, rec, ndcg = [], [], []
    for u, relevant in liked.items():
        top = np.argsort(-s[u])[:k]
        hits = np.array([i in relevant for i in top], dtype=float)
        prec.append(hits.mean())
        rec.append(hits.sum() / len(relevant))
        ndcg.append((hits * discount).sum() / discount[:min(len(relevant), k)].sum())
    return np.mean(prec), np.mean(rec), np.mean(ndcg)

all_u, all_i = np.arange(n_users)[:, None], np.arange(n_items)[None, :]
warm_items = np.isin(np.arange(n_items), train.item)
popularity = np.bincount(train.item, minlength=n_items).astype(float)
scorers = {
    "random":       rng.random((n_users, n_items)),
    "popularity":   np.tile(popularity, (n_users, 1)),
    "item CF":      mu + b_u.values[:, None] + b_i.values[None, :] + cf_resid,
    "matrix fact.": mf(all_u, all_i),
}
for name, sc in scorers.items():
    p, r, n = ranking_metrics(np.where(warm_items, sc, -np.inf), train, test_w)
    print(f"{name:<13} P@10 {p:.3f}  R@10 {r:.3f}  NDCG@10 {n:.3f}")

### Why offline metrics disagree with online tests

Even a well-chosen offline metric is a proxy. Logged data comes from the old system, so users could only click what it showed, and offline metrics reward models that imitate it. Clicks are not satisfaction, retention or revenue. A list is judged as a whole, so diversity and novelty matter. And a new model changes the very data it will later be trained on. Use offline metrics to shortlist models and catch regressions, then decide with an online A/B test on the outcome the product exists for (lesson 17).

To try the code on real data, this cell downloads MovieLens 100k: 100,000 ratings of 1,682 films by 943 users. Nothing later depends on it.

In [ ]:
# Downloads a model or data on first run
import io, urllib.request, zipfile

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())) as z:
    ml = pd.read_csv(z.open("ml-100k/u.data"), sep="\t",
                     names=["user", "item", "rating", "timestamp"])
ml[["user", "item"]] -= 1                          # ids start at 1 in the file
print(ml.shape, ml.user.nunique(), "users", ml.item.nunique(), "items")

### Hands-on: tune and blend the factorisation

1. Train `train_mf` with $k$ = 1, 2, 4, 8 and 16 factors and report the test RMSE. Which $k$ wins, and does it match the rank of 4 we planted?
2. Train with $k = 16$ and `reg=0.0`. Compare training and test RMSE.
3. Rank by the MF score plus $w \log(1 + \text{popularity})$ for $w$ = 0, 0.25, 0.5, 1 and 2, and report NDCG@10. Can a blend beat popularity alone (0.150)?

In [ ]:
# Your code here

## Two-tower retrieval

A **two-tower model** has two networks. The user tower turns what is known about a user (their id, recent history, context) into a vector; the item tower turns an item's id and content features into a vector in the same space. The score is the dot product. Because the towers meet only at that dot product, every item vector can be computed in advance and stored in an approximate nearest-neighbour index, exactly as in lesson 13. At request time you compute one user vector and search. That is why two-tower models are the standard candidate generator.

They train on implicit positives (this user chose this item) with **in-batch negatives**: in a batch of $B$ pairs, each user's own item is the positive and the other $B-1$ items in the batch act as negatives. With $s_{jl}$ the similarity of user $j$ and item $l$, and a temperature $\tau$, the loss is a softmax over each row of the $B \times B$ score matrix, the same contrastive loss that trains text and image embeddings:

$$
\mathcal{L} = -\frac{1}{B}\sum_{j=1}^{B} \log \frac{\exp(s_{jj}/\tau)}{\sum_{l=1}^{B} \exp(s_{jl}/\tau)}
$$

One correction matters a lot. Items enter batches in proportion to their popularity, so popular items serve as negatives far more often than they should and get pushed down. The **logQ correction**, described by Yi and colleagues at Google in 2019, subtracts the log of each item's sampling probability from its logit during training; at serving time you use the plain dot product. Our item tower also adds a genre embedding to the item id: a content feature that will matter for cold start.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, n_genres, dim=32):
        super().__init__()
        self.user_id = nn.Embedding(n_users, dim)
        self.item_id = nn.Embedding(n_items, dim)
        self.genre = nn.Embedding(n_genres, dim)   # a content feature
        self.user_mlp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.item_mlp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))

    def users(self, u):
        return F.normalize(self.user_mlp(self.user_id(u)), dim=-1)

    def items(self, i, g, use_id=True):
        x = self.genre(g) + (self.item_id(i) if use_id else 0)
        return F.normalize(self.item_mlp(x), dim=-1)

torch.manual_seed(0)
model = TwoTower(n_users, n_items, n_genres)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
U, I = torch.tensor(train.user.values), torch.tensor(train.item.values)
genre_t = torch.tensor(genre)
log_q = torch.tensor(np.log(np.bincount(train.item, minlength=n_items) / len(train)
                            + 1e-12), dtype=torch.float32)   # log sampling prob.
tau = 0.1                                          # softmax temperature

for epoch in range(6):
    total = 0.0
    for b in torch.randperm(len(U)).split(512):
        u, i = U[b], I[b]
        logits = model.users(u) @ model.items(i, genre_t[i]).T / tau   # (B, B)
        logits = logits - log_q[i]                 # logQ correction, per column
        loss = F.cross_entropy(logits, torch.arange(len(b)))   # diagonal = positive
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(b)
    print(f"epoch {epoch}: loss {total / len(U):.3f}")

The loss looks high because each row is a 512-way choice; what matters is how well the model ranks the test items.

In [ ]:
with torch.no_grad():
    user_vecs = model.users(torch.arange(n_users))             # (1000, 32)
    item_vecs = model.items(torch.arange(n_items), genre_t)    # (500, 32): index these
two_tower = (user_vecs @ item_vecs.T).numpy()

for name, sc in [("popularity", scorers["popularity"]),
                 ("matrix fact.", scorers["matrix fact."]),
                 ("two-tower", two_tower)]:
    p, r, n = ranking_metrics(np.where(warm_items, sc, -np.inf), train, test_w)
    print(f"{name:<13} P@10 {p:.3f}  R@10 {r:.3f}  NDCG@10 {n:.3f}")

## Ranking models

Retrieval returns a few hundred candidates; a **ranking model** then scores each one carefully. It can use features the towers cannot, because it sees the user and the item together: has this user watched this creator before, how well does the genre match their last week, what time is it, which device. Gradient-boosted trees (lesson 04) make a strong first ranker; large systems use deep networks that learn feature crosses and embeddings. Rankers usually predict several outcomes (click, watch time, like, purchase) and combine them with weights that encode business priorities.

## Cold start

**Cold start** is recommending to new users, or recommending new items, with no history. Collaborative filtering cannot score the 45 items released after our cut at all. The standard remedies:

- **Content features** describe items by category, creator, and text or image embeddings, so the item tower can place a new item before anyone has touched it.
- **Popularity and context** serve new users: what is popular in their region or on their device, then what they clicked earlier in this session. A few onboarding questions help too.
- **Exploration** deliberately shows new items to some users to learn about them quickly. This is the multi-armed bandit problem of lesson 25: balance exploiting what you know against exploring what you do not.

## Feedback loops, popularity bias and fairness

A recommender decides what people see, people can only respond to what they see, and tomorrow's model trains on those responses. Popular items get more exposure, so more clicks, so more exposure. Left alone, this **feedback loop** starves the long tail and can narrow what each person sees, the "filter bubble" (how strong that effect is in practice is still debated). The defences:

- log what was shown and with what probability, so later training can correct for exposure;
- reserve a little traffic for exploration;
- re-rank for diversity, for example by capping the items per genre or creator;
- track catalogue coverage and how exposure is spread, not just accuracy.

## LLMs and embeddings in recommendation

Text and image embeddings (lesson 13) are now common item features: they place a new item near similar old ones from its description alone, which helps cold start. LLMs can also rerank a short candidate list given a user's history, explain recommendations, and power conversational recommenders, although cost and latency usually confine them to small catalogues or the last stage. A research direction called **generative retrieval** gives each item a short code of discrete tokens (a semantic ID) and trains a sequence model to generate the code of the next item a user will want. Early results are promising, but two-tower retrieval plus a ranker remains the production workhorse.

### Hands-on: cold-start items and the logQ correction

1. Embed every item with `model.items(..., use_id=False)`, so the 45 new items are described by their genre alone. For each user, rank only the new items, and compute recall@10 and NDCG@10 against the test ratings of 4 or 5 on them. Compare with a random ranking.
2. Train a second two-tower model identical to the first but without the logQ correction, and evaluate it on warm items as above. What happens, and why?

In [ ]:
# Your code here

## Graphs: connected data

A graph is a set of nodes joined by edges. Edges can be directed (follows, pays) or undirected (is friends with), and weighted. Both can carry features: a user's age, an atom's element, a payment's amount. Three representations are common:

- an **edge list**, one row per edge, which is how graph data usually arrives;
- an **adjacency list** of each node's neighbours, fast to traverse and what networkx uses inside;
- an **adjacency matrix** $A$, with $A_{uv} = 1$ when there is an edge, which turns graph operations into matrix products and is stored sparse for large graphs.

### Classic algorithms that matter in ML

**PageRank** scores importance as the long-run share of time a random surfer spends at each node: at every step they follow a random edge with probability $d$, or jump to a random node otherwise.

$$
r_v = \frac{1-d}{n} + d \sum_{u \in \mathcal{N}(v)} \frac{r_u}{\deg(u)}
$$

**Shortest paths** give distances, such as how many hops an account is from a known fraudster. **Community detection** finds groups with many edges inside and few between; the Louvain method greedily maximises a score called modularity.

In [ ]:
import numpy as np
import networkx as nx

G = nx.karate_club_graph()
club = np.array([G.nodes[v]["club"] == "Officer" for v in G], dtype=int)
A = nx.to_numpy_array(G, weight=None)              # 34 x 34 adjacency matrix
print(G.number_of_nodes(), "members,", G.number_of_edges(), "friendships")
print("neighbours of node 0:", list(G[0])[:8], "...")
print("degree of nodes 0, 33, 11:", A.sum(1)[[0, 33, 11]].astype(int))

def pagerank(A, d=0.85, iters=100):
    """Power iteration: long-run share of time a random surfer spends at each node."""
    T = A / A.sum(1, keepdims=True)                # step to a random neighbour
    r = np.full(len(A), 1 / len(A))
    for _ in range(iters):
        r = (1 - d) / len(A) + d * T.T @ r
    return r

pr = pagerank(A)
same = np.allclose(pr, list(nx.pagerank(G, weight=None).values()), atol=1e-4)
print("highest PageRank:", np.argsort(-pr)[:4], "| matches networkx:", same)
print("shortest path from 16 to 25:", nx.shortest_path(G, 16, 25))
communities = nx.community.louvain_communities(G, seed=0)
purity = sum(np.bincount(club[list(c)]).max() for c in communities) / len(club)
print(f"{len(communities)} communities, sizes {[len(c) for c in communities]}, "
      f"purity {purity:.0%}")

Draw the graph, coloured by the club each member joined after the split.

In [ ]:
import matplotlib.pyplot as plt

pos = nx.spring_layout(G, seed=0)
plt.figure(figsize=(7, 5))
nx.draw(G, pos, node_color=club, cmap="coolwarm", with_labels=True, node_size=300)
plt.title("Zachary's karate club, coloured by the club each member joined")
plt.show()

## Node embeddings: word2vec on random walks

DeepWalk turns a graph into text. Walk the graph at random, treat each walk as a sentence of node ids, and train word2vec (lesson 07) on the sentences: nodes that turn up in the same walks get similar vectors. node2vec adds two parameters that bias the walks towards staying local or wandering far. Skip-gram word2vec is closely related to factorising a matrix of pointwise mutual information, so we can take the counting shortcut from that lesson: co-occurrence counts, PPMI, then the SVD.

In [ ]:
rng = np.random.default_rng(0)
neighbours = [list(G[v]) for v in G]
walks = []
for _ in range(20):                                # 20 walks of length 10 per node
    for start in G:
        walk = [start]
        for _ in range(9):
            nbrs = neighbours[walk[-1]]
            walk.append(nbrs[rng.integers(len(nbrs))])
        walks.append(walk)
print("one walk:", walks[0])

C = np.zeros((34, 34))                             # co-occurrence within 3 steps
for walk in walks:
    for j, v in enumerate(walk):
        for u in walk[max(0, j - 3):j] + walk[j + 1:j + 4]:
            C[v, u] += 1
ppmi = np.maximum(np.log(C * C.sum() / np.outer(C.sum(1), C.sum(0)) + 1e-12), 0)
U, S, _ = np.linalg.svd(ppmi)
emb = U[:, :4] * np.sqrt(S[:4])                    # 4-d node embeddings
emb /= np.linalg.norm(emb, axis=1, keepdims=True)

# label each member by the leader (0 or 33) whose embedding is closer
pred = (emb @ emb[33] > emb @ emb[0]).astype(int)
others = np.setdiff1d(np.arange(34), [0, 33])
print(f"accuracy on the other 32 members: {(pred[others] == club[others]).mean():.3f}")

## Graph neural networks

### Message passing

A **graph neural network** (GNN) learns node vectors from features and structure together. Each layer performs **message passing**: every node gathers its neighbours' vectors, aggregates them with a function that ignores order (sum, mean or max, because neighbours have no natural order), and updates its own vector with a learned transformation. After $L$ layers, a node's vector depends on everything within $L$ hops.

### A graph convolutional network from scratch

The graph convolutional network (GCN) of Kipf and Welling writes one layer as two matrix products:

$$
H^{(l+1)} = \sigma\left(\hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2} H^{(l)} W^{(l)}\right), \qquad \hat{A} = A + I
$$

$H^{(l)}$ holds one row per node, $W^{(l)}$ is an ordinary weight matrix, and multiplying by the normalised adjacency averages each node with its neighbours. The self-loops in $\hat{A}$ keep each node's own information, and dividing by the square roots of the degrees in $\hat{D}$ stops well-connected nodes from blowing up the scale. The karate club has no node features, so each node's input is a one-hot vector of its id. We label only the two leaders and train the network to classify everyone else: **semi-supervised node classification**, where the loss covers a few nodes but the graph connects them all.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

A_hat = torch.tensor(A, dtype=torch.float32) + torch.eye(34)   # add self-loops
deg = A_hat.sum(1)
A_norm = A_hat / torch.sqrt(deg[:, None] * deg[None, :])       # D^-1/2 A D^-1/2

class GCN(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.lin1 = nn.Linear(n_in, n_hidden)
        self.lin2 = nn.Linear(n_hidden, n_out)

    def forward(self, X, A_norm):
        h = torch.relu(A_norm @ self.lin1(X))      # transform, then mix neighbours
        return A_norm @ self.lin2(h)               # class scores for every node

X = torch.eye(34)                                  # no features: one-hot node ids
labels = torch.tensor(club)
labelled = torch.tensor([0, 33])                   # we only know the two leaders

def train_nodes(model, A_used, epochs=200):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epochs):
        loss = F.cross_entropy(model(X, A_used)[labelled], labels[labelled])
        opt.zero_grad(); loss.backward(); opt.step()
    pred = model(X, A_used).argmax(1).numpy()
    return (pred[others] == club[others]).mean(), others[pred[others] != club[others]]

torch.manual_seed(0)
acc, wrong = train_nodes(GCN(34, 16, 2), A_norm)
print(f"GCN accuracy {acc:.3f}, misclassified: {wrong}")
torch.manual_seed(0)
acc, _ = train_nodes(GCN(34, 16, 2), torch.eye(34))   # no edges: just an MLP
print(f"MLP accuracy {acc:.3f}")

### GraphSAGE, GAT and over-smoothing

## Where graphs show up

### Hands-on: watch over-smoothing happen

1. Generalise `GCN` to any number of layers (hidden size 16, ReLU between layers), train versions with 2, 4, 8 and 16 layers using `train_nodes`, and report their accuracy.
2. Without any training, take random features $H$ (34 by 16) and compute $\tilde{A}^L H$ for $L$ = 1, 2, 4, 8, 16 and 32, where $\tilde{A}$ is `A_norm`. Report the mean cosine similarity between different nodes' vectors.
3. Explain how the two results are connected.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/28-time-series-recsys-graphs.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.